# Fake News Detection System
**CS 6140 – Machine Learning | Northeastern University**  
**Team:** Ritik Pravin Mota · Steve George · Sia Sopori

---
This notebook implements a multi-model fake news detection pipeline:
1. **TF-IDF + Logistic Regression** — lightweight baseline
2. **LSTM** — sequential deep learning model
3. **RoBERTa** — fine-tuned transformer

Models are evaluated on the ISOT dataset and cross-tested on FakeNewsNet for generalizability.

In [1]:
# Install notebook dependencies required for all models and visualizations
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--break-system-packages",
    "torch",
    "transformers",
    "datasets",
    "evaluate",
    "tensorflow-cpu",
    "shap",
    "lime",
    "gradio",
    "accelerate",
])

0

In [2]:
# Import core libraries for data handling, modeling, and evaluation
import os
import pandas as pd
import numpy as np
import re
import pickle
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from torch.utils.data import Dataset
import evaluate
from sklearn.model_selection import train_test_split

/home/codespace/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Create output directory for all report figures
os.makedirs("figures", exist_ok=True)
print("Figures directory ready.")

Figures directory ready.


## 1. Data Loading & Preprocessing
We use the ISOT Fake News Dataset (44,898 articles, 2016-2017):
- 23,481 fake articles (52%) from sites flagged by PolitiFact
- 21,417 real articles (48%) from Reuters
Key preprocessing: Reuters byline removal, URL stripping, whitespace normalization. Split: 80% train / 10% val / 10% test.

In [4]:
# Load local ISOT datasets and build a cleaned labeled corpus
fake = pd.read_csv("Fake.csv/Fake.csv")
real = pd.read_csv("True.csv/True.csv")
fake["label"] = 0
real["label"] = 1
df = pd.concat([fake, real]).sample(frac=1, random_state=42).reset_index(drop=True)

def clean_text(text):
    text = re.sub(r'^.*?\(Reuters\)\s*-\s*', '', str(text))
    text = re.sub(r'featured image.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["content"] = (df["title"] + " " + df["text"]).apply(clean_text)
df = df[df["content"].str.strip() != ""].reset_index(drop=True)

X = df["content"]
y = df["label"]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print("Data ready.")

Data ready.


## 2. RoBERTa Fine-tuning
roberta-base (125M parameters) fine-tuned for binary classification.
Config: 3 epochs, batch size 8, grad accum 2, warmup 500 steps, fp16.

In [5]:
# Tokenize ISOT train/validation text for RoBERTa fine-tuning
tokenizer_roberta = AutoTokenizer.from_pretrained("roberta-base")
train_encodings = tokenizer_roberta(list(X_train), truncation=True, max_length=512)
val_encodings = tokenizer_roberta(list(X_val), truncation=True, max_length=512)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_roberta, return_tensors="pt")
print("Tokenization done.")

: 

In [6]:
# Wrap tokenizer outputs and labels in a torch Dataset for Trainer
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = int(self.labels[idx])
        return item

train_dataset = NewsDataset(train_encodings, y_train)
val_dataset = NewsDataset(val_encodings, y_val)
print("Datasets ready.")

Datasets ready.


In [7]:
# Initialize a fresh RoBERTa classifier head for binary labels
model_roberta = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
print("Model loaded.")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 503.46it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


In [8]:
# Set an alternate Hugging Face endpoint for model downloads
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
print("HF endpoint configured.")

HF endpoint configured.


In [1]:
# Configure and train RoBERTa with the original hyperparameters
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="roberta_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model_roberta,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

checkpoint_path = "checkpoint-2245"
if torch.cuda.is_available():
    if os.path.exists(checkpoint_path):
        trainer.train(resume_from_checkpoint=checkpoint_path)
    else:
        trainer.train()
else:
    print("CUDA unavailable: skipping trainer.train() and using current model weights.")

NameError: name 'evaluate' is not defined

In [ ]:
# Evaluate RoBERTa on validation split after training or checkpoint load
from sklearn.metrics import classification_report

if trainer is not None:
    predictions = trainer.predict(val_dataset)
    y_pred_roberta = np.argmax(predictions.predictions, axis=-1)
else:
    y_pred_roberta = predict_roberta_labels(X_val)

print(classification_report(y_val, y_pred_roberta, target_names=["Fake", "Real"]))

In [ ]:
import shutil

# Save the fine-tuned or fallback RoBERTa model and tokenizer locally
if trainer is not None:
    trainer.save_model("roberta_final")
else:
    model_roberta.save_pretrained("roberta_final")
tokenizer_roberta.save_pretrained("roberta_final")

# Zip artifacts for easy transfer
shutil.make_archive("roberta_final", 'zip', "roberta_final")
print("Model saved and zipped.")

In [ ]:
# Evaluate RoBERTa on the held-out ISOT test set
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

if trainer is not None:
    test_dataset = NewsDataset(
        tokenizer_roberta(list(X_test), truncation=True, max_length=512),
        y_test
    )
    predictions_test = trainer.predict(test_dataset)
    y_pred_test = np.argmax(predictions_test.predictions, axis=-1)
else:
    y_pred_test = predict_roberta_labels(X_test)

report_roberta_isot = classification_report(y_test, y_pred_test, target_names=["Fake", "Real"], output_dict=True)
print(classification_report(y_test, y_pred_test, target_names=["Fake", "Real"]))

cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("RoBERTa — ISOT Test Set")
plt.savefig("figures/confusion_matrix_roberta.png", bbox_inches="tight", dpi=150)
plt.show()

## 3. TF-IDF + Logistic Regression & LSTM
TF-IDF baseline: 50K features, bigrams, English stopwords.
LSTM: Embedding(128) -> SpatialDropout -> LSTM(128) -> Dense(64) -> Sigmoid.
Early stopping with patience=2.

In [ ]:
# Verify datasets package version used for FakeNewsNet loading
import datasets
print(datasets.__version__)

In [ ]:
# Fix: import load_dataset (required for FakeNewsNet cross-dataset evaluation)
from datasets import load_dataset

In [ ]:
# Load the FakeNewsNet-compatible benchmark dataset from Hugging Face
dataset2 = load_dataset("GonzaloA/fake_news")
print(dataset2)

In [ ]:
# Build FakeNewsNet evaluation dataframe and cleaned text fields
df_fnn = pd.DataFrame(dataset2['test'])
df_fnn["content"] = (df_fnn["title"] + " " + df_fnn["text"]).apply(clean_text)
df_fnn = df_fnn[df_fnn["content"].str.strip() != ""].reset_index(drop=True)
X_fnn = df_fnn["content"]
y_fnn = df_fnn["label"]
print(df_fnn['label'].value_counts())
print(df_fnn.head(3))

In [ ]:
# Check label mapping
print(df_fnn[df_fnn['label']==0]['title'].head(2))
print("---")
print(df_fnn[df_fnn['label']==1]['title'].head(2))

In [ ]:
# List local repository artifacts for reproducibility checks
print(os.listdir("."))

In [ ]:
# Train TF-IDF + Logistic Regression baseline and evaluate on FakeNewsNet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)

lr2 = LogisticRegression(max_iter=1000)
lr2.fit(X_train_tfidf, y_train)

X_fnn_tfidf = tfidf.transform(X_fnn)
y_pred_lr_fnn = lr2.predict(X_fnn_tfidf)

report_tfidf_fnn = classification_report(y_fnn, y_pred_lr_fnn, target_names=["Fake", "Real"], output_dict=True)
print("TF-IDF + LR on FakeNewsNet:")
print(classification_report(y_fnn, y_pred_lr_fnn, target_names=["Fake", "Real"]))

In [ ]:
# Train the LSTM text classifier on tokenized ISOT sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB = 50000
MAX_LEN = 512

lstm_tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
lstm_tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(lstm_tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, truncating='post')
X_val_seq   = pad_sequences(lstm_tokenizer.texts_to_sequences(X_val),   maxlen=MAX_LEN, truncating='post')

lstm_model = Sequential([
    Embedding(MAX_VOCAB, 128),
    SpatialDropout1D(0.2),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history = lstm_model.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_val_seq, y_val),
    callbacks=[early_stop]
)

In [ ]:
# Evaluate LSTM transfer performance on FakeNewsNet
X_fnn_seq = pad_sequences(lstm_tokenizer.texts_to_sequences(X_fnn), maxlen=MAX_LEN, truncating='post')
y_pred_lstm_fnn = (lstm_model.predict(X_fnn_seq) > 0.5).astype(int).ravel()

report_lstm_fnn = classification_report(y_fnn, y_pred_lstm_fnn, target_names=["Fake", "Real"], output_dict=True)
print("LSTM on FakeNewsNet:")
print(classification_report(y_fnn, y_pred_lstm_fnn, target_names=["Fake", "Real"]))

In [ ]:
# Evaluate RoBERTa transfer performance on FakeNewsNet
if trainer is not None:
    fnn_encodings = tokenizer_roberta(list(X_fnn), truncation=True, max_length=512)
    fnn_dataset = NewsDataset(fnn_encodings, y_fnn)
    predictions_fnn = trainer.predict(fnn_dataset)
    y_pred_fnn = np.argmax(predictions_fnn.predictions, axis=-1)
else:
    y_pred_fnn = predict_roberta_labels(X_fnn)

report_roberta_fnn = classification_report(y_fnn, y_pred_fnn, target_names=["Fake", "Real"], output_dict=True)
print("RoBERTa on FakeNewsNet:")
print(classification_report(y_fnn, y_pred_fnn, target_names=["Fake", "Real"]))

## 4. Model Evaluation — ISOT Test Set
Evaluating all three models on the held-out ISOT test set (4,489 articles).
Confusion matrices show true/false positive breakdown per model.

In [ ]:
# Evaluate TF-IDF + LR on the held-out ISOT test split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_test_tfidf = tfidf.transform(X_test)
y_pred_lr_test = lr2.predict(X_test_tfidf)

report_tfidf_isot = classification_report(y_test, y_pred_lr_test, target_names=['Fake', 'Real'], output_dict=True)
print('=== TF-IDF + Logistic Regression — ISOT Test Set ===')
print(classification_report(y_test, y_pred_lr_test, target_names=['Fake', 'Real']))

cm_lr = confusion_matrix(y_test, y_pred_lr_test)
ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Fake', 'Real']).plot(cmap='Greens')
plt.title('TF-IDF + LR — ISOT Test Set')
plt.tight_layout()
plt.savefig("figures/confusion_matrix_tfidf.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Evaluate LSTM on the held-out ISOT test split
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_test_seq = pad_sequences(
    lstm_tokenizer.texts_to_sequences(X_test),
    maxlen=MAX_LEN, truncating='post'
)
y_pred_lstm_test = (lstm_model.predict(X_test_seq) > 0.5).astype(int).ravel()

report_lstm_isot = classification_report(y_test, y_pred_lstm_test, target_names=['Fake', 'Real'], output_dict=True)
print('=== LSTM — ISOT Test Set ===')
print(classification_report(y_test, y_pred_lstm_test, target_names=['Fake', 'Real']))

cm_lstm = confusion_matrix(y_test, y_pred_lstm_test)
ConfusionMatrixDisplay(confusion_matrix=cm_lstm, display_labels=['Fake', 'Real']).plot(cmap='Oranges')
plt.title('LSTM — ISOT Test Set')
plt.tight_layout()
plt.savefig("figures/confusion_matrix_lstm.png", bbox_inches="tight", dpi=150)
plt.show()

## 5. LSTM Training Curves
Accuracy and loss per epoch to verify convergence and early stopping behavior.

In [ ]:
# Plot LSTM training and validation curves for convergence diagnostics
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='o')
axes[0].set_title('LSTM — Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='o')
axes[1].set_title('LSTM — Loss per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('LSTM Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("figures/lstm_training_curves.png", bbox_inches="tight", dpi=150)
plt.show()

## 6. Model Comparison Summary
Consolidated results across both evaluation datasets.
Key finding: all three models achieve identical 97% F1 on FakeNewsNet despite RoBERTa's apparent advantage on ISOT.

In [ ]:
# Build a unified summary table from computed evaluation reports
df_results = pd.DataFrame(
    [
        {
            'Model': 'TF-IDF + Logistic Regression',
            'ISOT Accuracy': report_tfidf_isot['accuracy'],
            'ISOT F1': report_tfidf_isot['weighted avg']['f1-score'],
            'FakeNewsNet Accuracy': report_tfidf_fnn['accuracy'],
            'FakeNewsNet F1': report_tfidf_fnn['weighted avg']['f1-score'],
        },
        {
            'Model': 'LSTM',
            'ISOT Accuracy': report_lstm_isot['accuracy'],
            'ISOT F1': report_lstm_isot['weighted avg']['f1-score'],
            'FakeNewsNet Accuracy': report_lstm_fnn['accuracy'],
            'FakeNewsNet F1': report_lstm_fnn['weighted avg']['f1-score'],
        },
        {
            'Model': 'RoBERTa',
            'ISOT Accuracy': report_roberta_isot['accuracy'],
            'ISOT F1': report_roberta_isot['weighted avg']['f1-score'],
            'FakeNewsNet Accuracy': report_roberta_fnn['accuracy'],
            'FakeNewsNet F1': report_roberta_fnn['weighted avg']['f1-score'],
        },
    ]
)

for col in ['ISOT Accuracy', 'ISOT F1', 'FakeNewsNet Accuracy', 'FakeNewsNet F1']:
    df_results[col] = (df_results[col] * 100).round(2).astype(str) + '%'

print('=== Model Comparison Summary ===')
df_results

## 7. Explainability — SHAP & LIME
SHAP: global feature importance across all predictions.
LIME: instance-level explanations for individual articles.
Fake news signals: emotional language, clickbait, sensational terms.
Real news signals: formal attribution, precise dates, official terminology.

In [ ]:
# Generate and save SHAP global feature-importance summary
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "shap"])
import shap
import matplotlib.pyplot as plt

explainer = shap.LinearExplainer(lr2, X_train_tfidf, feature_perturbation="interventional")
shap_values = explainer(tfidf.transform(X_test[:100]))

shap.summary_plot(shap_values, feature_names=tfidf.get_feature_names_out(), max_display=20, show=False)
plt.savefig("figures/shap_summary.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Generate and save a LIME explanation for a fake-news sample
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "lime"])
from lime.lime_text import LimeTextExplainer
import matplotlib.pyplot as plt

explainer_lime = LimeTextExplainer(class_names=["Fake", "Real"])

def predict_proba(texts):
    return lr2.predict_proba(tfidf.transform(texts))

sample = X_test.iloc[5]
exp = explainer_lime.explain_instance(sample, predict_proba, num_features=10)
fig = exp.as_pyplot_figure()
fig.savefig("figures/lime_fake.png", bbox_inches="tight", dpi=150)
plt.close()
exp.show_in_notebook()

In [ ]:
# Generate and save a LIME explanation for a real-news sample
real_idx = y_test[y_test == 1].index[0]
sample_real = X_test.loc[real_idx]

exp_real = explainer_lime.explain_instance(sample_real, predict_proba, num_features=10)
fig = exp_real.as_pyplot_figure()
fig.savefig("figures/lime_real.png", bbox_inches="tight", dpi=150)
plt.close()
exp_real.show_in_notebook()

In [ ]:
# Install Gradio for the final web interface
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "gradio"])

## 8. Web Interface — Gradio Deployment
Real-time article classification using all three models simultaneously.

In [ ]:
# Inspect local workspace outputs and artifacts
print(os.listdir("."))

In [ ]:
# Sanity-check key object types before deployment
print(type(tfidf))
print(type(lr2))
print(type(lstm_tokenizer))
print(type(lstm_model))
print(type(model_roberta))
print(type(tokenizer_roberta))

In [ ]:
# Import Gradio and sequence utilities for interactive inference
import gradio as gr
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Build a multi-model prediction function and Gradio interface
def predict_news(text):
    cleaned = clean_text(text)
    
    tfidf_input = tfidf.transform([cleaned])
    lr_prob = lr2.predict_proba(tfidf_input)[0]
    lr_label = "Fake" if lr_prob[0] > 0.5 else "Real"
    lr_confidence = float(max(lr_prob))
    
    lstm_seq = pad_sequences(lstm_tokenizer.texts_to_sequences([cleaned]), maxlen=MAX_LEN, truncating='post')
    lstm_prob = lstm_model.predict(lstm_seq, verbose=0)[0][0]
    lstm_label = "Fake" if lstm_prob < 0.5 else "Real"
    lstm_confidence = float(1 - lstm_prob) if lstm_prob < 0.5 else float(lstm_prob)
    
    device = next(model_roberta.parameters()).device
    inputs = tokenizer_roberta(cleaned, return_tensors="pt", truncation=True, max_length=512, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model_roberta(**inputs)
    roberta_prob = torch.softmax(outputs.logits, dim=1)[0]
    roberta_label = "Fake" if roberta_prob[0] > roberta_prob[1] else "Real"
    roberta_confidence = float(max(roberta_prob))
    
    return {
        f"TF-IDF + LR: {lr_label}": lr_confidence,
        f"LSTM: {lstm_label}": lstm_confidence,
        f"RoBERTa: {roberta_label}": roberta_confidence,
    }

demo = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=10, placeholder="Paste a news article here..."),
    outputs=gr.Label(num_top_classes=3),
    title="Fake News Detector",
    description="Paste a news article and see predictions from three models — TF-IDF + LR, LSTM, and RoBERTa.",
    examples=[
        ["BREAKING: Hillary Clinton Arrested for Treason - Watch the Video Now!!!"],
        ["The Federal Reserve raised interest rates by 25 basis points on Wednesday, citing continued concerns about inflation, according to officials who spoke at a press conference."]
    ]
)
print("Gradio interface configured.")

In [ ]:
# Inspect local RoBERTa export directory if available
if os.path.exists("roberta_final"):
    print(os.listdir("roberta_final"))
else:
    print("roberta_final directory not found yet.")

In [ ]:
# Load a local RoBERTa checkpoint if present, then launch the Gradio app
from transformers import AutoTokenizer, AutoModelForSequenceClassification

if os.path.exists("roberta_final") and os.path.isdir("roberta_final"):
    tokenizer_roberta = AutoTokenizer.from_pretrained("roberta_final")
    model_roberta = AutoModelForSequenceClassification.from_pretrained("roberta_final")
    model_roberta.eval()
    print("RoBERTa loaded from local checkpoint.")
else:
    print("Using in-memory RoBERTa model from earlier cells.")

demo.launch(share=True, prevent_thread_lock=True)